# A2A 클라이언트와 Writer — 실습본

원본: `Chapter02/02-a2a/02_a2a_client.py`  
슬라이드: `99_pdf/Chapter02/day01/3-3-a2a-protocol-understanding.pdf`

## 학습 목표

- Agent Card 조회·메시지 전송·역할 분리
- 원본 코드를 CodeMate와 함께 읽고 실행 결과를 점검한다.


## 코드 흐름

Agent Card → SendMessageRequest → Researcher 응답 → Writer LLM

**읽을 때 확인할 점:** Researcher가 모의 검색을 사용할 수 있다. Writer의 글도 근거와 사실을 검토한다.


## 실행 전 확인

별도 터미널에서 01_a2a_server.py를 포트 8001로 먼저 실행한다. 이 셀은 로컬 HTTP와 LLM에 접속한다.

setup 셀은 `asyncio.run(main())`의 Jupyter 루프 충돌을 피하도록 스크립트 전용 `__main__` 분기를 건너뛰게 합니다. 원본 코드 셀에서 함수를 정의한 뒤, 아래의 별도 `await main()` 셀을 실행합니다. 아래 원본 코드 셀은 실행을 직접 선택해야 하는 `manual-run` 셀입니다. 실행 준비 셀은 프로젝트 루트를 찾고 원본 `.py`의 경로를 `__file__`에 설정합니다.


In [ ]:
from pathlib import Path
import os
import sys

_repo_root = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "Chapter02").is_dir()),
    None,
)
if _repo_root is None:
    raise FileNotFoundError("저장소 안에서 이 노트북을 열어 주세요.")
os.chdir(_repo_root)
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
__file__ = str(_repo_root / "Chapter02/02-a2a/02_a2a_client.py")
# Jupyter에서는 스크립트 전용 __main__ 분기를 실행하지 않습니다.
__name__ = "notebook"


## 원본 코드

다음 셀에는 원본 `.py` 전체가 그대로 들어 있습니다.


In [ ]:
"""
A2A 클라이언트: 다른 Agent에게 작업을 위임하기

실행 순서:
  1. 터미널 1: python 01_a2a_server.py
  2. 터미널 2: python 02_a2a_client.py
"""

import asyncio
import uuid

import a2a.types as t
import httpx
from a2a.client.base_client import BaseClient
from a2a.client.card_resolver import A2ACardResolver
from a2a.client.client import ClientConfig
from a2a.client.transports.jsonrpc import JsonRpcTransport
from a2a.utils.constants import PROTOCOL_VERSION_1_0, VERSION_HEADER
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage

from config.llm import get_llm

load_dotenv()

RESEARCHER_URL = 'http://localhost:8001'

async def call_researcher(query: str) -> str:
    """A2A 프로토콜로 Researcher Agent에게 조사를 요청한다.

    A2A-Version 헤더를 명시해야 서버가 1.x 형식 요청임을 인식한다.
    헤더가 없으면 서버는 레거시(0.3) 버전으로 가정하여 거부한다.
    """
    headers = {VERSION_HEADER: PROTOCOL_VERSION_1_0}
    async with httpx.AsyncClient(headers=headers, timeout=60.0) as http:
        # 1. Agent Card 조회: 에이전트가 어떤 기능을 제공하는지 확인
        # TODO: A2ACardResolver를 활용해 agent card를 조회해주세요.
        resolver = A2ACardResolver(http, RESEARCHER_URL)
        card = await resolver.get_agent_card()

        # 2. JSON-RPC 트랜스포트로 클라이언트 생성
        # TODO: http 클라이언트와 card 정보를 포함해 A2A 클라이언트를 생성해 주세요.
        transport = JsonRpcTransport(http, card, f'{RESEARCHER_URL}/')
        client = BaseClient(card, ClientConfig(), transport, [])

        # 3. 메시지 전송
        # TODO: t.SendMessageRequest를 생성해주세요.
        req = t.SendMessageRequest(
            message=t.Message(
                role=t.Role.ROLE_USER,
                parts=[t.Part(text=query)],
                message_id=str(uuid.uuid4())
            )
        )
        
        async for response in client.send_message(req):
            # StreamResponse.message: 에이전트가 반환한 Message 객체
            if response.HasField('message'):
                return ''.join(
                    part.text for part in response.message.parts if part.text
                )

    return '응답을 받지 못했습니다.'

async def main():
    # ── Agent Card 확인 ────────────────────────────────────────────────────────
    async with httpx.AsyncClient() as http:
        resp = await http.get(f'{RESEARCHER_URL}/.well-known/agent-card.json')
        card = resp.json()

    print('=== Researcher Agent Card ===')
    print(f'이름   : {card["name"]}')
    print(f'설명   : {card["description"]}')
    print(f'스킬   : {[s["name"] for s in card.get("skills", [])]}')
    print()

    # ── Researcher에게 조사 위임 ───────────────────────────────────────────────
    topic = 'LangGraph Multi-Agent 시스템의 실제 활용 사례'
    print(f'=== Researcher에게 위임: "{topic}" ===')
    research_result = await call_researcher(topic)
    print(f'리서치 결과:\n{research_result}\n')

    # ── Writer가 리서치 결과로 글 작성 ────────────────────────────────────────
    # Writer는 직접 LLM을 쓰지만, 리서치 작업은 Researcher에게 위임했다.
    # → 이종 Agent 간 역할 분리가 A2A의 핵심
    print('=== Writer가 블로그 포스트 작성 ===')
    llm = get_llm()
    response = llm.invoke([
        SystemMessage(content='너는 전문 작가야. 리서치 결과를 바탕으로 블로그 포스트를 작성해줘.'),
        HumanMessage(content=f'리서치 결과:\n{research_result}\n\n이 내용으로 블로그 포스트를 작성해줘.'),
    ])
    print(response.content)


if __name__ == '__main__':
    asyncio.run(main())


## 클라이언트 실행

서버 연결 조건을 확인한 뒤 다음 셀을 직접 실행하세요. 네트워크 및 LLM 호출이 발생합니다.


In [ ]:
await main()


## CodeMate와 함께 검토

> Message의 role, parts, message_id를 설명하고 반환된 텍스트가 Writer 입력이 되는 경로를 추적해 줘. 원본은 바꾸지 말고 검토 내용을 먼저 말해 줘.

제안된 변경은 어떤 문제를 해결하는지, 어떤 결과로 확인할지 설명해 보세요.

## 확인 질문

1. 서버가 꺼져 있으면 어디서 먼저 실패하는가?
2. Researcher와 Writer의 책임은?
